# Geliştirilmiş Dengeli XGBoost Üretim Tahmin Pipeline'ı (Direct Horizon)

Bu notebook,dört sorunu doğrudan hedefler:

1. **Adaptif guardrail**: Seri volatilitesine ve horizon'a göre dinamik alt/üst log-değişim sınırları.
2. **Daha kontrollü düşüş bandı**: Önceki sabit `0.55` alt sınırı kaldırıldı; seri volatilitesine göre genelde `0.72–0.90` bandında daha korumacı bir alt seviye sınırı uygulanır.
3. **Recursive yerine direct horizon modelleri**: `t+1`, `t+2`, `t+3` için ayrı modeller kurulur. Böylece tahmin üstüne tahmin binmesi azaltılır.
4. **Future row group-stat iyileştirmesi**: Ürün/şehir bazlı yıl-medyanları, son gözlenen tarihe kadar olan grup tarihi üzerinden **projeksiyon** ile üretilir.



In [ ]:

# =========================
# 1) Kurulum ve importlar
# =========================
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu notebook'u calistirmak icin xgboost gerekli: pip install xgboost") from exc

RANDOM_STATE = 42


def find_repo_root():
    current = Path.cwd().resolve()
    for base in [current, *current.parents]:
        if (base / "veri").exists() and (base / "models").exists():
            return base
    raise FileNotFoundError("Repo kok dizini bulunamadi; veri/ ve models/ klasorleri gorunmuyor.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "veri"
OUTPUT_DIR = REPO_ROOT / "models" / "uretim"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

PATHS = {
    "Meyve": DATA_DIR / "Detayli_Meyve_Tam_Yatay.xlsx",
    "Sebze": DATA_DIR / "Detayli_Sebze_Tam_Yatay.xlsx",
    "Tahil": DATA_DIR / "Detayli_Tahil_Verisi_Yatay.xlsx",
    "Iklim": DATA_DIR / "Turkiye_81_Il_Tarimsal_Iklim_2013_2024.xlsx",
}

FORECAST_YEARS = [2025, 2026, 2027]
HORIZONS = [1, 2, 3]

print("Veri klasoru:", DATA_DIR)
print("Uretim model cikti klasoru:", OUTPUT_DIR)
for label, path in PATHS.items():
    print(label, "->", path.name, "VAR" if path.exists() else "YOK")


In [ ]:
# =========================
# 2) Yardımcı fonksiyonlar
# =========================
def pick_first_existing(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Bulunamadı. Adaylar: {candidates}")
    return None

def normalize_text(s):
    if pd.isna(s):
        return np.nan
    return str(s).strip()

def safe_to_datetime(series):
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    out = np.zeros_like(y_true, dtype=float)
    out[mask] = 2.0 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]
    return float(np.mean(out) * 100)

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return float(np.sum(np.abs(y_true - y_pred)) / denom * 100)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def ensure_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clip_series(x, low_q=0.01, high_q=0.99):
    x = pd.Series(x)
    if x.notna().sum() < 5:
        return x
    lo = x.quantile(low_q)
    hi = x.quantile(high_q)
    return x.clip(lo, hi)

def make_native_categorical(df, cat_cols):
    out = df.copy()
    for c in cat_cols:
        if c in out.columns:
            out[c] = out[c].astype("category")
    return out

def historical_group_stat_by_year(df, group_cols, value_col, stat="median"):
    """Return group statistic using only years strictly before the row year.

    This avoids leakage from other rows in the same year. The previous implementation
    used expanding().shift(1) on the current row order; for product/city groups with
    many cities per year, that could let earlier rows from the same year enter the
    feature value.
    """
    if stat not in {"median", "max", "mean"}:
        raise ValueError("Bilinmeyen stat")

    yearly = (
        df.groupby(group_cols + ["Yil"], as_index=False)[value_col]
          .agg(stat)
          .sort_values(group_cols + ["Yil"])
    )

    def _calc(series):
        shifted = series.shift(1)
        if stat == "median":
            return shifted.expanding().median()
        if stat == "max":
            return shifted.expanding().max()
        return shifted.expanding().mean()

    yearly["__hist_stat"] = yearly.groupby(group_cols, group_keys=False)[value_col].transform(_calc)
    merged = df[group_cols + ["Yil"]].merge(
        yearly[group_cols + ["Yil", "__hist_stat"]],
        on=group_cols + ["Yil"],
        how="left",
        sort=False,
    )
    return merged["__hist_stat"].reset_index(drop=True)

def safe_last(values, default=np.nan):
    vals = pd.Series(values).dropna()
    return vals.iloc[-1] if len(vals) else default


In [ ]:
# =========================
# 3) Veri yükleme
# =========================
def load_production_data(path, category_name):
    df = pd.read_excel(path)

    city_col    = pick_first_existing(df, ["Sehir_Adi", "İl", "Il", "Sehir"])
    product_col = pick_first_existing(df, ["Urun_Adi", "Ürün Adı", "Urun", "Ürün"])
    year_col    = pick_first_existing(df, ["Yil", "Yıl", "Year"])
    target_col  = pick_first_existing(df, ["Üretim Miktarı (Ton)", "Uretim Miktari (Ton)", "Üretim", "Uretim", "Miktar"])
    method_col  = pick_first_existing(df, ["Uretim_Yontemi", "Üretim Yöntemi", "Yontem"], required=False)
    area_col    = pick_first_existing(df, [
        "Alan (Dekar)",
        "Ekim Alanı (Dekar)",
        "Hasat Alanı (Dekar)",
        "Meyve Veren Yaşta Ağaç Sayısı (Adet Sayısı)",
        "Meyve Veren Yaşta Ağaç Sayısı",
        "Ağaç Sayısı",
        "Agac Sayisi"
    ], required=False)

    rename_map = {
        city_col: "Sehir_Adi",
        product_col: "Urun_Adi",
        year_col: "Yil",
        target_col: "Uretim_Ton",
    }
    if method_col:
        rename_map[method_col] = "Uretim_Yontemi"
    if area_col:
        rename_map[area_col] = "Alan_Proxy"

    df = df.rename(columns=rename_map).copy()

    if "Uretim_Yontemi" not in df.columns:
        df["Uretim_Yontemi"] = "Bilinmiyor"
    if "Alan_Proxy" not in df.columns:
        df["Alan_Proxy"] = np.nan

    keep_cols = ["Sehir_Adi", "Urun_Adi", "Uretim_Yontemi", "Yil", "Uretim_Ton", "Alan_Proxy"]
    df = df[keep_cols].copy()

    df["Sehir_Adi"] = df["Sehir_Adi"].map(normalize_text)
    df["Urun_Adi"] = df["Urun_Adi"].map(normalize_text)
    df["Uretim_Yontemi"] = df["Uretim_Yontemi"].map(normalize_text)
    df["Yil"] = pd.to_numeric(df["Yil"], errors="coerce")
    df["Uretim_Ton"] = pd.to_numeric(df["Uretim_Ton"], errors="coerce")
    df["Alan_Proxy"] = pd.to_numeric(df["Alan_Proxy"], errors="coerce")
    df["Kategori"] = category_name

    df = df.dropna(subset=["Sehir_Adi", "Urun_Adi", "Yil", "Uretim_Ton"]).copy()
    df["Yil"] = df["Yil"].astype(int)
    df["Uretim_Ton"] = df["Uretim_Ton"].clip(lower=0)

    df = (df.groupby(["Kategori", "Sehir_Adi", "Urun_Adi", "Uretim_Yontemi", "Yil"], as_index=False)
            .agg({"Uretim_Ton": "sum", "Alan_Proxy": "sum"}))

    df["Series_ID"] = (
        df["Kategori"].astype(str) + " | " +
        df["Sehir_Adi"].astype(str) + " | " +
        df["Urun_Adi"].astype(str) + " | " +
        df["Uretim_Yontemi"].astype(str)
    )
    return df

def load_climate_data(path):
    df = pd.read_excel(path)

    city_col = pick_first_existing(df, ["Sehir", "Sehir_Adi", "İl", "Il"])
    date_col = pick_first_existing(df, ["Tarih", "Date"])

    df = df.rename(columns={city_col: "Sehir_Adi", date_col: "Tarih"}).copy()
    df["Tarih"] = safe_to_datetime(df["Tarih"])
    df = df.dropna(subset=["Tarih"]).copy()

    df["Sehir_Adi"] = df["Sehir_Adi"].map(normalize_text)
    df["Yil"] = df["Tarih"].dt.year

    numeric_candidates = [
        "Sicaklik_Ort_C", "Yagis_mm", "Ruzgar_Hizi", "Toprak_Nemi_%",
        "Nem_%", "Guneslenme_Suresi", "Basinc_hPa"
    ]
    numeric_cols = [c for c in numeric_candidates if c in df.columns]
    df = ensure_numeric(df, numeric_cols)

    agg_map = {}
    for c in numeric_cols:
        agg_map[c] = "sum" if "Yagis" in c else "mean"

    annual = df.groupby(["Sehir_Adi", "Yil"], as_index=False).agg(agg_map)

    for c in numeric_cols:
        city_mean = annual.groupby("Sehir_Adi")[c].transform("mean")
        annual[f"{c}_anom"] = annual[c] - city_mean

    return annual

df_meyve = load_production_data(PATHS["Meyve"], "Meyve")
df_sebze = load_production_data(PATHS["Sebze"], "Sebze")
df_tahil = load_production_data(PATHS["Tahil"], "Tahil")
df_iklim = load_climate_data(PATHS["Iklim"])

full_prod = pd.concat([df_meyve, df_sebze, df_tahil], ignore_index=True)
full_prod = full_prod.sort_values(["Series_ID", "Yil"]).reset_index(drop=True)

print("Üretim veri boyutu:", full_prod.shape)
print("Benzersiz seri:", full_prod["Series_ID"].nunique())
print("Yıl aralığı:", int(full_prod["Yil"].min()), "-", int(full_prod["Yil"].max()))

full_prod.head()

In [ ]:
# =========================
# 4) Tarihsel özellik seti
#    - Recursive hedef yerine origin-year feature frame
# =========================
def build_base_feature_frame(prod_df, climate_df):
    df = prod_df.copy().sort_values(["Series_ID", "Yil"]).reset_index(drop=True)
    df = df.merge(climate_df, on=["Sehir_Adi", "Yil"], how="left")

    g = df.groupby("Series_ID", group_keys=False)

    # Mevcut seviye ve geçmiş
    df["current_y"] = df["Uretim_Ton"]
    df["current_log_y"] = np.log1p(df["current_y"])

    df["lag1_y"] = g["Uretim_Ton"].shift(1)
    df["lag2_y"] = g["Uretim_Ton"].shift(2)
    df["lag3_y"] = g["Uretim_Ton"].shift(3)

    df["lag1_log"] = np.log1p(df["lag1_y"])
    df["lag2_log"] = np.log1p(df["lag2_y"])
    df["lag3_log"] = np.log1p(df["lag3_y"])

    df["recent_mean_3"] = g["Uretim_Ton"].transform(lambda s: s.rolling(3, min_periods=1).mean())
    df["recent_median_3"] = g["Uretim_Ton"].transform(lambda s: s.rolling(3, min_periods=1).median())
    df["recent_std_3"] = g["Uretim_Ton"].transform(lambda s: s.rolling(3, min_periods=2).std())
    df["recent_mean_log_3"] = g["current_log_y"].transform(lambda s: s.rolling(3, min_periods=1).mean())

    df["recent_yoy_1"] = (df["current_y"] - df["lag1_y"]) / (df["lag1_y"] + 1e-6)
    df["recent_yoy_2"] = (df["lag1_y"] - df["lag2_y"]) / (df["lag2_y"] + 1e-6)
    df["delta_log_hist_1"] = df["current_log_y"] - df["lag1_log"]
    df["delta_log_hist_2"] = df["lag1_log"] - df["lag2_log"]

    df["area_current"] = df["Alan_Proxy"]
    df["area_lag1"] = g["Alan_Proxy"].shift(1)
    df["area_lag2"] = g["Alan_Proxy"].shift(2)
    df["area_yoy"] = (df["area_current"] - df["area_lag1"]) / (df["area_lag1"] + 1e-6)

    df["series_count_so_far"] = g.cumcount() + 1
    df["first_year"] = g["Yil"].transform("min")
    df["age_of_series"] = df["Yil"] - df["first_year"]

    # Tarihsel (leakage azaltılmış) group stat'lar
    df["urun_hist_medyan"] = historical_group_stat_by_year(df, ["Kategori", "Urun_Adi"], "Uretim_Ton", "median")
    df["sehir_hist_medyan"] = historical_group_stat_by_year(df, ["Kategori", "Sehir_Adi"], "Uretim_Ton", "median")
    df["series_hist_medyan"] = historical_group_stat_by_year(df, ["Series_ID"], "Uretim_Ton", "median")
    df["series_hist_max"] = historical_group_stat_by_year(df, ["Series_ID"], "Uretim_Ton", "max")

    for c in ["recent_yoy_1", "recent_yoy_2", "delta_log_hist_1", "delta_log_hist_2", "area_yoy"]:
        if c in df.columns:
            df[c] = clip_series(df[c], 0.01, 0.99)

    return df

base_df = build_base_feature_frame(full_prod, df_iklim)
print("Base feature frame boyutu:", base_df.shape)
base_df.head()


In [ ]:
# =========================
# 5) Direct horizon veri seti ve group-stat projeksiyonu
# =========================
def build_group_median_history(prod_df, group_cols):
    out = {}
    grp = (prod_df.groupby(group_cols + ["Yil"], as_index=False)["Uretim_Ton"]
           .median()
           .sort_values(group_cols + ["Yil"]))
    key_cols = group_cols
    for keys, sub in grp.groupby(key_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        out[keys] = sub[["Yil", "Uretim_Ton"]].reset_index(drop=True)
    return out

urun_median_map = build_group_median_history(full_prod, ["Kategori", "Urun_Adi"])
sehir_median_map = build_group_median_history(full_prod, ["Kategori", "Sehir_Adi"])

def estimate_group_future_median(history_map, key, origin_year, horizon):
    hist = history_map.get(key)
    if hist is None or len(hist) == 0:
        return np.nan

    hist = hist[hist["Yil"] <= origin_year].copy().sort_values("Yil")
    if len(hist) == 0:
        return np.nan

    y0 = float(hist.iloc[-1]["Uretim_Ton"])
    if len(hist) == 1:
        return y0

    log_hist = np.log1p(hist["Uretim_Ton"].astype(float))
    d1 = log_hist.diff().dropna()

    if len(d1) == 0:
        return y0

    recent = d1.tail(min(3, len(d1)))
    mean_delta = float(recent.mean())

    # Damping: horizon uzadıkça grup projeksiyonunu daha muhafazakâr yap
    scale_map = {1: 1.00, 2: 1.70, 3: 2.25}
    damp_map = {1: 0.90, 2: 0.80, 3: 0.72}
    cum_delta = mean_delta * scale_map.get(horizon, float(horizon)) * damp_map.get(horizon, 0.70)

    vol = float(recent.std(ddof=0)) if len(recent) >= 2 else abs(mean_delta) * 0.35
    lo = max(-0.30, recent.quantile(0.10) - 0.02 - 0.20 * vol)
    hi = min( 0.30, recent.quantile(0.90) + 0.02 + 0.20 * vol)
    cum_delta = float(np.clip(cum_delta, lo * horizon, hi * horizon))

    return float(np.expm1(np.log1p(max(y0, 0.0)) + cum_delta))



def cached_group_future_median(history_map, cache, key, origin_year, horizon):
    cache_key = (tuple(key), int(origin_year), int(horizon))
    if cache_key not in cache:
        cache[cache_key] = estimate_group_future_median(history_map, tuple(key), int(origin_year), int(horizon))
    return cache[cache_key]

def make_direct_frames(base_df, horizons=(1, 2, 3)):
    frames = {}
    g = base_df.groupby("Series_ID", group_keys=False)

    for h in horizons:
        dfh = base_df.copy()
        dfh["Forecast_Horizon"] = h
        dfh["Forecast_Year"] = dfh["Yil"] + h
        dfh["target_y"] = g["Uretim_Ton"].shift(-h)
        dfh["target_delta_log"] = np.log1p(dfh["target_y"]) - np.log1p(dfh["current_y"])

        # Future-year group-stat projections.
        # Cache avoids recomputing the same product/city-year-horizon value for many rows.
        urun_cache = {}
        sehir_cache = {}
        dfh["urun_yil_medyan_est"] = [
            cached_group_future_median(
                urun_median_map,
                urun_cache,
                (kategori, urun),
                yil,
                h,
            )
            for kategori, urun, yil in zip(dfh["Kategori"], dfh["Urun_Adi"], dfh["Yil"])
        ]
        dfh["sehir_yil_medyan_est"] = [
            cached_group_future_median(
                sehir_median_map,
                sehir_cache,
                (kategori, sehir),
                yil,
                h,
            )
            for kategori, sehir, yil in zip(dfh["Kategori"], dfh["Sehir_Adi"], dfh["Yil"])
        ]

        # Eğitim için yeterli history
        dfh = dfh.dropna(subset=["target_y", "lag1_y", "lag2_y", "current_y"]).copy()
        dfh = dfh[dfh["series_count_so_far"] >= 4].copy()
        frames[h] = dfh.reset_index(drop=True)

    return frames

direct_frames = make_direct_frames(base_df, HORIZONS)

for h, d in direct_frames.items():
    print(f"Horizon {h}: {d.shape}")


In [ ]:
# =========================
# 6) Özellik listesi
# =========================
base_features = [
    "Kategori", "Sehir_Adi", "Urun_Adi", "Uretim_Yontemi",
    "Yil", "Forecast_Year", "Forecast_Horizon",
    "age_of_series", "series_count_so_far",
    "current_y", "current_log_y",
    "lag1_y", "lag2_y", "lag3_y",
    "lag1_log", "lag2_log", "lag3_log",
    "recent_mean_3", "recent_median_3", "recent_std_3", "recent_mean_log_3",
    "recent_yoy_1", "recent_yoy_2",
    "delta_log_hist_1", "delta_log_hist_2",
    "area_current", "area_lag1", "area_lag2", "area_yoy",
    "urun_yil_medyan_est", "sehir_yil_medyan_est",
    "urun_hist_medyan", "sehir_hist_medyan",
    "series_hist_medyan", "series_hist_max"
]

climate_features = [c for c in base_df.columns if c in [
    "Sicaklik_Ort_C", "Yagis_mm", "Ruzgar_Hizi", "Toprak_Nemi_%",
    "Nem_%", "Guneslenme_Suresi", "Basinc_hPa",
    "Sicaklik_Ort_C_anom", "Yagis_mm_anom", "Ruzgar_Hizi_anom", "Toprak_Nemi_%_anom",
    "Nem_%_anom", "Guneslenme_Suresi_anom", "Basinc_hPa_anom"
]]

features = [c for c in base_features + climate_features if c in base_df.columns or c in ["Forecast_Year", "Forecast_Horizon", "urun_yil_medyan_est", "sehir_yil_medyan_est"]]

sample_train = pd.concat(list(direct_frames.values()), ignore_index=True)
cat_cols = [c for c in features if c in ["Kategori", "Sehir_Adi", "Urun_Adi", "Uretim_Yontemi"]]
num_cols = [c for c in features if c not in cat_cols]

print("Feature sayısı:", len(features))
print("Kategorik:", cat_cols)

In [ ]:
# =========================
# 7) Model, adaptif guardrail ve post-process
# =========================
def build_xgb_model():
    return XGBRegressor(
        n_estimators=1400,  # Full model; long walk-forward can take several minutes.
        learning_rate=0.025,
        max_depth=6,
        min_child_weight=10,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.25,
        reg_lambda=2.5,
        objective="reg:squarederror",
        tree_method="hist",
        max_cat_to_onehot=16,
        enable_categorical=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=60
    )

def direct_baseline_delta(row, horizon):
    vals = [row.get("delta_log_hist_1", np.nan), row.get("delta_log_hist_2", np.nan)]
    vals = [float(v) for v in vals if pd.notna(v)]
    if len(vals) == 0:
        return 0.0

    mean_delta = float(np.mean(vals))
    scale_map = {1: 1.00, 2: 1.65, 3: 2.15}
    damp_map  = {1: 0.95, 2: 0.82, 3: 0.72}
    return mean_delta * scale_map.get(horizon, float(horizon)) * damp_map.get(horizon, 0.70)

def compute_adaptive_guardrails(hist_series, horizon=1):
    hist = pd.Series(hist_series).dropna().astype(float)
    if len(hist) <= horizon + 1:
        base = 0.12 * horizon
        return (-base, base, 0.10)

    log_hist = np.log1p(hist)
    d = log_hist.diff(horizon).dropna()

    if len(d) < 3:
        base = 0.12 * horizon
        return (-base, base, float(d.std(ddof=0)) if len(d) else 0.10)

    vol = float(d.std(ddof=0))
    q10 = float(d.quantile(0.10))
    q90 = float(d.quantile(0.90))

    pad = 0.02 + 0.35 * min(vol, 0.25)

    lo = q10 - pad
    hi = q90 + pad

    # Horizon ve volatiliteye göre dinamik sert limitler
    lo_floor = -(0.14 * horizon + 0.05 * min(vol, 0.25) * horizon)
    hi_cap   = +(0.16 * horizon + 0.06 * min(vol, 0.25) * horizon)

    lo = max(lo, lo_floor, -0.42)
    hi = min(hi, hi_cap,  0.48)

    return float(lo), float(hi), vol

def compute_level_bounds(recent_values, horizon, trend_signal, vol):
    recent = pd.Series(recent_values).dropna().astype(float)
    if len(recent) < 2:
        return 0.0, np.inf

    vol_ratio = min(1.0, vol / 0.20)

    # Düşüş senaryosunda eski 0.55 yerine daha kontrollü alt bant
    if trend_signal < 0:
        lower_mult = np.clip(0.88 - 0.10 * (horizon - 1) - 0.10 * vol_ratio, 0.72, 0.90)
        upper_mult = np.clip(1.08 + 0.06 * vol_ratio + 0.05 * (horizon - 1), 1.08, 1.28)
    else:
        lower_mult = np.clip(0.84 - 0.06 * (horizon - 1) - 0.08 * vol_ratio, 0.74, 0.90)
        upper_mult = np.clip(1.24 + 0.08 * vol_ratio + 0.07 * (horizon - 1), 1.20, 1.42)

    lower_level = max(0.0, float(recent.min() * lower_mult))
    upper_level = float(recent.max() * upper_mult)
    return lower_level, upper_level

def blend_delta(delta_ml, delta_base, horizon):
    weights = {1: (0.70, 0.30), 2: (0.65, 0.35), 3: (0.60, 0.40)}
    w_ml, w_base = weights.get(horizon, (0.60, 0.40))
    return float(w_ml * delta_ml + w_base * delta_base)

def post_process_prediction(row, delta_hat, hist_values, horizon):
    lo, hi, vol = compute_adaptive_guardrails(hist_values, horizon=horizon)
    delta_hat = float(np.clip(delta_hat, lo, hi))

    trend_signal = float(row.get("delta_log_hist_1", 0.0)) if pd.notna(row.get("delta_log_hist_1", np.nan)) else 0.0

    # Trend yönünü tamamen tersine çeviren sert hareketleri yumuşat
    if trend_signal > 0.05 and delta_hat < (-0.12 * horizon):
        delta_hat = -0.12 * horizon
    if trend_signal < -0.05 and delta_hat > (0.10 * horizon):
        delta_hat = 0.10 * horizon

    y_prev = max(float(row["current_y"]), 0.0)
    y_hat = float(np.expm1(np.log1p(y_prev) + delta_hat))
    y_hat = max(0.0, y_hat)

    recent = pd.Series(hist_values).dropna().tail(3)
    lower_level, upper_level = compute_level_bounds(recent, horizon, trend_signal, vol)
    y_hat = float(np.clip(y_hat, lower_level, upper_level))
    y_hat = max(0.0, y_hat)

    return y_hat, delta_hat, lo, hi, vol


In [ ]:
# =========================
# 8) Horizon bazlı walk-forward backtest
# =========================
def train_single_horizon_model(train_df_h, features, cat_cols):
    work = train_df_h.copy()

    for c in num_cols:
        if c in work.columns:
            work[c] = pd.to_numeric(work[c], errors="coerce")

    holdout_origin = int(work["Yil"].max())
    dev = work[work["Yil"] < holdout_origin].copy()
    val = work[work["Yil"] == holdout_origin].copy()

    if len(dev) == 0 or len(val) == 0:
        dev = work.copy()
        val = work.tail(min(1000, len(work))).copy()

    medians = {}
    for c in num_cols:
        if c in dev.columns:
            med = float(dev[c].median()) if dev[c].notna().sum() else 0.0
            medians[c] = med
            dev[c] = dev[c].fillna(med)
            val[c] = val[c].fillna(med)

    dev = make_native_categorical(dev, cat_cols)
    val = make_native_categorical(val, cat_cols)

    model = build_xgb_model()
    model.fit(dev[features], dev["target_delta_log"], eval_set=[(val[features], val["target_delta_log"])], verbose=False)
    return model, medians

def evaluate_direct_backtest(direct_frames, features, cat_cols):
    all_metrics = []
    all_preds = []

    history_lookup = {
        sid: g.sort_values("Yil").copy()
        for sid, g in full_prod.groupby("Series_ID")
    }

    for horizon, dfh in direct_frames.items():
        origin_years = sorted(dfh["Yil"].unique())
        origin_years = [y for y in origin_years if y >= (min(origin_years) + 2)]

        for origin_year in origin_years:
            print(f"Backtest fold basliyor: horizon={horizon}, origin_year={origin_year}", flush=True)
            fold_train = dfh[dfh["Yil"] < origin_year].copy()
            fold_test = dfh[dfh["Yil"] == origin_year].copy()

            if len(fold_train) < 500 or len(fold_test) == 0:
                continue

            model, fold_medians = train_single_horizon_model(fold_train, features, cat_cols)

            for c in num_cols:
                if c in fold_test.columns:
                    fold_test[c] = pd.to_numeric(fold_test[c], errors="coerce").fillna(fold_medians.get(c, 0.0))
            fold_test = make_native_categorical(fold_test, cat_cols)

            pred_delta_ml = model.predict(fold_test[features])

            pred_level = []
            pred_delta_final = []
            lo_list, hi_list = [], []

            for i, row in fold_test.reset_index(drop=True).iterrows():
                sid = row["Series_ID"]
                hist = history_lookup[sid]
                hist_vals = hist.loc[hist["Yil"] <= origin_year, "Uretim_Ton"].tolist()

                delta_base = direct_baseline_delta(row, horizon)
                delta_hat = blend_delta(float(pred_delta_ml[i]), delta_base, horizon)

                y_hat, delta_used, lo, hi, vol = post_process_prediction(row, delta_hat, hist_vals, horizon)

                pred_level.append(y_hat)
                pred_delta_final.append(delta_used)
                lo_list.append(lo)
                hi_list.append(hi)

            y_true = fold_test["target_y"].values
            y_pred = np.array(pred_level, dtype=float)

            all_metrics.append({
                "origin_year": int(origin_year),
                "forecast_year": int(origin_year + horizon),
                "horizon": int(horizon),
                "R2": float(r2_score(y_true, y_pred)),
                "MAE": float(mean_absolute_error(y_true, y_pred)),
                "RMSE": rmse(y_true, y_pred),
                "SMAPE_%": smape(y_true, y_pred),
                "WAPE_%": wape(y_true, y_pred),
            })

            fold_out = fold_test[[
                "Series_ID", "Kategori", "Sehir_Adi", "Urun_Adi", "Uretim_Yontemi",
                "Yil", "Forecast_Year", "target_y"
            ]].copy()
            fold_out = fold_out.rename(columns={"Yil": "Origin_Yil", "target_y": "Gercek_Uretim"})
            fold_out["Tahmin"] = y_pred
            fold_out["delta_log_guard_lo"] = lo_list
            fold_out["delta_log_guard_hi"] = hi_list
            fold_out["horizon"] = horizon
            all_preds.append(fold_out)
            print(f"Backtest fold tamamlandi: horizon={horizon}, origin_year={origin_year}, WAPE={wape(y_true, y_pred):.2f}", flush=True)

    metrics_df = pd.DataFrame(all_metrics)
    preds_df = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    return metrics_df, preds_df

metrics_df, wf_preds = evaluate_direct_backtest(direct_frames, features, cat_cols)
metrics_df.head()


In [ ]:
# Walk-forward özet
if len(metrics_df):
    display(metrics_df.sort_values(["forecast_year", "horizon"]).head(20))
    print("Horizon bazında ortalama metrikler:")
    display(metrics_df.groupby("horizon")[["R2", "MAE", "RMSE", "SMAPE_%", "WAPE_%"]].mean())
else:
    print("Backtest metriği üretilemedi. Veri yılları ve horizon kapsaması kontrol edilmeli.")

In [ ]:
# =========================
# 9) Son modelleri eğit
# =========================
final_models = {}
final_medians = {}

for horizon, dfh in direct_frames.items():
    model_h, medians_h = train_single_horizon_model(dfh, features, cat_cols)
    final_models[horizon] = model_h
    final_medians[horizon] = medians_h
    print(f"Horizon {horizon} model hazır. Best iteration:", getattr(model_h, "best_iteration", None))

In [ ]:
# =========================
# 10) Future row üretimi (iyileştirilmiş)
# =========================
def get_last_known_climate(city, year=None):
    sub = df_iklim[df_iklim["Sehir_Adi"] == city].sort_values("Yil")
    if len(sub) == 0:
        return {}
    if year is not None:
        sub = sub[sub["Yil"] <= year]
        if len(sub) == 0:
            return {}
    return sub.iloc[-1].to_dict()

def make_future_origin_row(series_hist, target_year):
    hist = series_hist.sort_values("Yil").copy()
    last = hist.iloc[-1]
    origin_year = int(last["Yil"])
    horizon = int(target_year - origin_year)

    current_y = float(last["Uretim_Ton"])
    lag1_y = float(hist.iloc[-2]["Uretim_Ton"]) if len(hist) >= 2 else np.nan
    lag2_y = float(hist.iloc[-3]["Uretim_Ton"]) if len(hist) >= 3 else np.nan
    lag3_y = float(hist.iloc[-4]["Uretim_Ton"]) if len(hist) >= 4 else np.nan

    current_log_y = np.log1p(current_y)
    lag1_log = np.log1p(lag1_y) if pd.notna(lag1_y) else np.nan
    lag2_log = np.log1p(lag2_y) if pd.notna(lag2_y) else np.nan
    lag3_log = np.log1p(lag3_y) if pd.notna(lag3_y) else np.nan

    recent_vals = pd.Series([current_y, lag1_y, lag2_y], dtype=float).dropna()

    out = {
        "Series_ID": last["Series_ID"],
        "Kategori": last["Kategori"],
        "Sehir_Adi": last["Sehir_Adi"],
        "Urun_Adi": last["Urun_Adi"],
        "Uretim_Yontemi": last["Uretim_Yontemi"],
        "Yil": origin_year,
        "Forecast_Year": target_year,
        "Forecast_Horizon": horizon,

        "current_y": current_y,
        "current_log_y": current_log_y,
        "lag1_y": lag1_y,
        "lag2_y": lag2_y,
        "lag3_y": lag3_y,
        "lag1_log": lag1_log,
        "lag2_log": lag2_log,
        "lag3_log": lag3_log,

        "recent_mean_3": float(recent_vals.mean()) if len(recent_vals) else np.nan,
        "recent_median_3": float(recent_vals.median()) if len(recent_vals) else np.nan,
        "recent_std_3": float(recent_vals.std()) if len(recent_vals) >= 2 else 0.0,
        "recent_mean_log_3": float(np.log1p(recent_vals).mean()) if len(recent_vals) else np.nan,

        "recent_yoy_1": ((current_y - lag1_y) / (lag1_y + 1e-6)) if pd.notna(lag1_y) else 0.0,
        "recent_yoy_2": ((lag1_y - lag2_y) / (lag2_y + 1e-6)) if pd.notna(lag1_y) and pd.notna(lag2_y) else 0.0,
        "delta_log_hist_1": (current_log_y - lag1_log) if pd.notna(lag1_log) else 0.0,
        "delta_log_hist_2": (lag1_log - lag2_log) if pd.notna(lag1_log) and pd.notna(lag2_log) else 0.0,

        "area_current": float(last["Alan_Proxy"]) if pd.notna(last["Alan_Proxy"]) else np.nan,
        "area_lag1": float(hist.iloc[-2]["Alan_Proxy"]) if len(hist) >= 2 and pd.notna(hist.iloc[-2]["Alan_Proxy"]) else np.nan,
        "area_lag2": float(hist.iloc[-3]["Alan_Proxy"]) if len(hist) >= 3 and pd.notna(hist.iloc[-3]["Alan_Proxy"]) else np.nan,
        "area_yoy": 0.0,

        "series_count_so_far": len(hist),
        "first_year": int(hist["Yil"].min()),
        "age_of_series": int(origin_year - hist["Yil"].min()),

        # Gelecek grup özetleri: history bazlı projeksiyon
        "urun_yil_medyan_est": estimate_group_future_median(
            urun_median_map, (last["Kategori"], last["Urun_Adi"]), origin_year, horizon
        ),
        "sehir_yil_medyan_est": estimate_group_future_median(
            sehir_median_map, (last["Kategori"], last["Sehir_Adi"]), origin_year, horizon
        ),

        "urun_hist_medyan": float(hist["Uretim_Ton"].median()),
        "sehir_hist_medyan": float(full_prod[
            (full_prod["Kategori"] == last["Kategori"]) &
            (full_prod["Sehir_Adi"] == last["Sehir_Adi"]) &
            (full_prod["Yil"] <= origin_year)
        ]["Uretim_Ton"].median()),
        "series_hist_medyan": float(hist["Uretim_Ton"].median()),
        "series_hist_max": float(hist["Uretim_Ton"].max()),
    }

    climate_info = get_last_known_climate(last["Sehir_Adi"], year=origin_year)
    if climate_info:
        for c in climate_features:
            if c in climate_info:
                out[c] = climate_info[c]

    return out


In [ ]:
# =========================
# 11) Gelecek tahminleri üret (Direct horizon)
# =========================
history_map = {
    sid: g.sort_values("Yil").copy()
    for sid, g in full_prod.groupby("Series_ID")
}

future_rows = []

total_series = len(history_map)

for series_index, (sid, hist) in enumerate(history_map.items(), start=1):
    if series_index == 1 or series_index % 500 == 0 or series_index == total_series:
        print(f"Future tahmin progress: {series_index}/{total_series}", flush=True)

    last_year = int(hist["Yil"].max())

    for target_year in FORECAST_YEARS:
        horizon = int(target_year - last_year)
        if horizon not in final_models:
            continue

        row = make_future_origin_row(hist, target_year)
        row_df = pd.DataFrame([row])

        medians_h = final_medians[horizon]

        for c in features:
            if c not in row_df.columns:
                row_df[c] = np.nan

        for c in num_cols:
            if c in row_df.columns:
                row_df[c] = pd.to_numeric(row_df[c], errors="coerce").fillna(medians_h.get(c, 0.0))
        row_df = make_native_categorical(row_df, cat_cols)

        delta_ml = float(final_models[horizon].predict(row_df[features])[0])
        delta_base = direct_baseline_delta(row_df.iloc[0], horizon)
        delta_hat = blend_delta(delta_ml, delta_base, horizon)

        hist_vals = hist["Uretim_Ton"].tolist()
        y_hat, delta_used, lo, hi, vol = post_process_prediction(row_df.iloc[0], delta_hat, hist_vals, horizon)

        future_rows.append({
            "Kategori": row["Kategori"],
            "Sehir_Adi": row["Sehir_Adi"],
            "Urun_Adi": row["Urun_Adi"],
            "Uretim_Yontemi": row["Uretim_Yontemi"],
            "Yil": target_year,
            "Tahmini_Uretim_Ton": y_hat,
            "Origin_Yil": last_year,
            "Forecast_Horizon": horizon,
            "delta_log_used": delta_used,
            "delta_log_guard_lo": lo,
            "delta_log_guard_hi": hi,
            "vol_proxy": vol,
        })

future_df = pd.DataFrame(future_rows)
future_df = future_df.sort_values(["Kategori", "Sehir_Adi", "Urun_Adi", "Uretim_Yontemi", "Yil"]).reset_index(drop=True)

print("Future tahmin boyutu:", future_df.shape)
future_df.head()


In [ ]:
# =========================
# 12) Çıktıları kaydet
# =========================
output_path = OUTPUT_DIR / "Dengeli_XGBoost_DirectHorizon_2025_2027_Tahminler.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for cat in ["Meyve", "Sebze", "Tahil"]:
        sub = future_df[future_df["Kategori"] == cat].copy()
        sub.to_excel(writer, sheet_name=cat[:31], index=False)

    metrics_df.to_excel(writer, sheet_name="WalkForward_Metrikler", index=False)
    wf_preds.to_excel(writer, sheet_name="WalkForward_Tahminler", index=False)

print("Excel models/uretim klasorune kaydedildi:", output_path)


In [ ]:
# =========================
# 13) Seri grafik fonksiyonu
# =========================
def plot_series(category, city, product, method=None):
    hist = full_prod[
        (full_prod["Kategori"] == category) &
        (full_prod["Sehir_Adi"] == city) &
        (full_prod["Urun_Adi"] == product)
    ].copy()

    fut = future_df[
        (future_df["Kategori"] == category) &
        (future_df["Sehir_Adi"] == city) &
        (future_df["Urun_Adi"] == product)
    ].copy()

    if method is not None:
        hist = hist[hist["Uretim_Yontemi"] == method].copy()
        fut = fut[fut["Uretim_Yontemi"] == method].copy()

    if hist.empty and fut.empty:
        print("Seri bulunamadı.")
        return

    plt.figure(figsize=(12, 5))

    if not hist.empty:
        hist = hist.sort_values("Yil")
        plt.plot(hist["Yil"], hist["Uretim_Ton"], marker="o", label="Gerçek Üretim")

    if not fut.empty:
        fut = fut.sort_values("Yil")
        plt.plot(fut["Yil"], fut["Tahmini_Uretim_Ton"], marker="o", linestyle="--", label="Direct Horizon Tahmin")

    title = f"{category} | {city} | {product}"
    if method is not None:
        title += f" | {method}"

    plt.title(title)
    plt.xlabel("Yıl")
    plt.ylabel("Üretim (Ton)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# Örnek:
# plot_series("Meyve", "Adana", "Armut")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# AYARLAR VE KLASÖR OLUŞTURMA
# =========================
GRAPH_DIR = Path.cwd() / "model_grafikleri"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# Görselleştirme teması
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

# =========================
# 1) KATEGORİ BAZLI GERÇEK VS TAHMİN (ZAMAN SERİSİ)
# =========================
categories = ["Meyve", "Sebze", "Tahil"]
for cat in categories:
    cat_dir = GRAPH_DIR / cat
    cat_dir.mkdir(parents=True, exist_ok=True)

    # Geçmiş Veri
    hist = full_prod[full_prod["Kategori"] == cat]
    hist_year = hist.groupby("Yil")["Uretim_Ton"].sum().reset_index()

    # Gelecek Tahmin Verisi
    fut = future_df[future_df["Kategori"] == cat]
    fut_year = fut.groupby("Yil")["Tahmini_Uretim_Ton"].sum().reset_index()

    plt.figure(figsize=(10, 6))
    plt.plot(hist_year["Yil"], hist_year["Uretim_Ton"], marker="o", label="Gerçek Üretim (Geçmiş)")
    plt.plot(fut_year["Yil"], fut_year["Tahmini_Uretim_Ton"], marker="o", linestyle="--", label="Model Tahmini (Gelecek)")
    
    plt.title(f"{cat} Kategorisi: Toplam Üretim Projeksiyonu")
    plt.xlabel("Yıl")
    plt.ylabel("Toplam Üretim (Ton)")
    plt.legend()
    plt.savefig(cat_dir / f"{cat}_uretim_trend.png", bbox_inches="tight")
    plt.close()

# =========================
# 2) WALK-FORWARD BACKTEST ANALİZLERİ
# =========================
if not wf_preds.empty:
    df_plot = wf_preds.copy()
    
    # Hatalı sütun isimlerini düzeltiyoruz
    # Kodun üst kısımlarında 'Gercek_Uretim' olarak isimlendirilmişti.
    target_col = "Gercek_Uretim" 
    predict_col = "Tahmin"

    # A) Gerçek vs Tahmin Scatter Plot
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=df_plot, x=target_col, y=predict_col, alpha=0.5)
    
    # İdeal Tahmin Çizgisi (y=x)
    max_val = max(df_plot[target_col].max(), df_plot[predict_col].max())
    min_val = min(df_plot[target_col].min(), df_plot[predict_col].min())
    plt.plot([min_val, max_val], [min_val, max_val], color='red', lw=2, linestyle='--')
    
    plt.title("Backtest Performansı: Gerçek vs Tahmin")
    plt.xlabel("Gerçek Üretim (Ton)")
    plt.ylabel("Tahmin Edilen Üretim (Ton)")
    plt.savefig(GRAPH_DIR / "backtest_scatter_plot.png")
    plt.close()

    # B) Residual (Hata) Plot
    plt.figure(figsize=(10, 6))
    df_plot["Residual"] = df_plot[target_col] - df_plot[predict_col]
    sns.scatterplot(data=df_plot, x=predict_col, y="Residual", alpha=0.5)
    plt.axhline(0, color='black', lw=2, linestyle='--')
    
    plt.title("Hata Dağılımı (Residual Plot)")
    plt.xlabel("Tahmin Edilen Değer")
    plt.ylabel("Hata (Gerçek - Tahmin)")
    plt.savefig(GRAPH_DIR / "backtest_residual_plot.png")
    plt.close()

# =========================
# 3) SHAP DEĞERLERİ (ÖZELLİK ÖNEMİ)
# =========================
try:
    import shap
    # İlk horizon modelini örnek alıyoruz
    model_to_explain = final_models[1]
    # SHAP için kategorik verileri sayısal koda çevirmek gerekebilir
    # XGBoost native kategorik desteği olsa da SHAP bazen ham veri bekler
    X_sample = direct_frames[1][features].sample(min(500, len(direct_frames[1])))
    
    # SHAP explainer
    explainer = shap.TreeExplainer(model_to_explain)
    shap_values = explainer.shap_values(X_sample)

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title("Özelliklerin Tahmine Etki Düzeyleri (SHAP Summary)")
    plt.savefig(GRAPH_DIR / "shap_ozellik_onemi.png", bbox_inches="tight")
    plt.close()
    print("SHAP grafiği başarıyla kaydedildi.")
except Exception as e:
    print(f"SHAP grafiği oluşturulurken hata: {e}")

# =========================
# 4) LEARNING CURVE (EĞİTİM GEÇMİŞİ)
# =========================
for horizon, model in final_models.items():
    try:
        results = model.evals_result()
        plt.figure(figsize=(8, 5))
        plt.plot(results["validation_0"]["rmse"], label="Eğitim (Train)")
        plt.plot(results["validation_1"]["rmse"], label="Doğrulama (Val)")
        plt.title(f"Model Eğitim Geçmişi - Horizon {horizon}")
        plt.xlabel("İterasyon")
        plt.ylabel("RMSE Hata")
        plt.legend()
        plt.savefig(GRAPH_DIR / f"learning_curve_h{horizon}.png")
        plt.close()
    except:
        pass

print(f"\nİşlem tamamlandı. Tüm grafikler '{GRAPH_DIR}' klasörüne kaydedildi.")